In [1]:
import pandas as pd

In [2]:
!unzip ds_job_salaries

Archive:  ds_job_salaries.zip
  inflating: Data Science Job Salary dataset/datascience_salaries.csv  


In [28]:
ds_salaries = pd.read_csv('/content/Data Science Job Salary dataset/datascience_salaries.csv')
ds_salaries.head()

,Unnamed: 0,job_title,job_type,experience_level,location,salary_currency,salary
0,0,Data scientist,Full Time,Senior,New York City,USD,149000
1,2,Data scientist,Full Time,Senior,Boston,USD,120000
2,3,Data scientist,Full Time,Senior,London,USD,68000
3,4,Data scientist,Full Time,Senior,Boston,USD,120000
4,5,Data scientist,Full Time,Senior,New York City,USD,149000


**What You Will Learn**

Advanced techniques for data normalization, reduction, and aggregation.
Skills in gathering, exploring, integrating, and cleaning data using Python.
Proficiency in using Pandas for complex data manipulation.


**Your Task**


*   Download and import the Data Science Job Salary dataset.

*   Normalize the ‘salary’ column using Min-Max normalization which scales all salary values between 0 and 1.

*   Implement dimensionality reduction like Principal Component Analysis (PCA) or t-SNE to reduce the number of features (columns) in the dataset.

*   Group the dataset by the ‘experience_level’ column and calculate the average and median salary for each experience level (e.g., Junior, Mid-level, Senior).


**Hint** :
As a reminder, normalization is crucial when dealing with data that has different ranges. For example, salary data might have a wide range (e.g., from $20,000 to $200,000). By scaling the data using Min-Max normalization, you make sure that all salary values fall within a consistent range (0 to 1). This is particularly helpful when the data is going to be used in machine learning models, as some algorithms (like k-nearest neighbors or neural networks) perform better when features are normalized. It ensures that no single salary dominates the learning process, making the analysis more balanced.

Dimensionality reduction helps simplify complex datasets by reducing the number of variables under consideration. This can make the data more manageable and help avoid the curse of dimensionality—a phenomenon where machine learning models struggle when dealing with high-dimensional data.
PCA, for instance, helps in retaining the most important information (variance) from the dataset while reducing noise and redundancy.
It can also speed up the training process for models and help in visualizing data in fewer dimensions.

Aggregating data helps in understanding trends within subgroups of the dataset.
Calculating average and median salaries for each experience level gives insights into the compensation distribution and disparities across different job levels. This kind of aggregation can help in answering business questions like “How does salary evolve with experience?” or “What is the salary distribution for senior-level roles?”

In [29]:
# checking dispersion of numeric values:
ds_salaries.describe()


,Unnamed: 0,salary
count,1171.000000,1171.000000
mean,931.620837,64836.037575
std,647.445266,32551.767046
min,0.000000,30000.000000
25%,364.500000,45000.000000
50%,815.000000,63000.000000
75%,1504.500000,68000.000000
max,2259.000000,228000.000000


In [30]:
# checking completes of data
ds_salaries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1171 entries, 0 to 1170
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1171 non-null   int64 
 1   job_title         1171 non-null   object
 2   job_type          1171 non-null   object
 3   experience_level  1171 non-null   object
 4   location          1171 non-null   object
 5   salary_currency   1171 non-null   object
 6   salary            1171 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 64.2+ KB


In [31]:
# before normalizing salary, to check for outliers and replace them:
q25 = ds_salaries['salary'].quantile(0.25)
q75 = ds_salaries['salary'].quantile(0.75)
print(f"25-percentile: {q25}")
print(f"75-percentile: {q75}")

IQR = q75 - q25
lower_bound = q25 - 1.5 * IQR
upper_bound = q75 + 1.5 * IQR
print(f"upper boundary: {upper_bound}")
print(f"lower boundary: {lower_bound}")

print(f"95-percentile: {ds_salaries['salary'].quantile(0.95)}")

25-percentile: 45000.0
75-percentile: 68000.0
upper boundary: 102500.0
lower boundary: 10500.0
95-percentile: 137500.0


In [34]:
# ONLY 5% ARE ABOVE 137.5k, WE COULD USE THIS AS A CAP FOR ALL iqr-BASED OUTLIERS:
cap = ds_salaries['salary'].quantile(0.95)
ds_salaries['salary_cap'] = ds_salaries['salary'].apply(lambda x: cap if x > cap else x)

In [35]:
ds_salaries.head()

,Unnamed: 0,job_title,job_type,experience_level,location,salary_currency,salary,salary_cap
0,0,Data scientist,Full Time,Senior,New York City,USD,149000,137500.0
1,2,Data scientist,Full Time,Senior,Boston,USD,120000,120000.0
2,3,Data scientist,Full Time,Senior,London,USD,68000,68000.0
3,4,Data scientist,Full Time,Senior,Boston,USD,120000,120000.0
4,5,Data scientist,Full Time,Senior,New York City,USD,149000,137500.0


In [36]:
# NOW normalizing salary column using Min-Max normalization which scales all salary values between 0 and 1
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
ds_salaries['salary_cap_normalized'] = scaler.fit_transform(ds_salaries[['salary_cap']])

In [37]:
ds_salaries.head()

,Unnamed: 0,job_title,job_type,experience_level,location,salary_currency,salary,salary_cap,salary_cap_normalized
0,0,Data scientist,Full Time,Senior,New York City,USD,149000,137500.0,1.000000
1,2,Data scientist,Full Time,Senior,Boston,USD,120000,120000.0,0.837209
2,3,Data scientist,Full Time,Senior,London,USD,68000,68000.0,0.353488
3,4,Data scientist,Full Time,Senior,Boston,USD,120000,120000.0,0.837209
4,5,Data scientist,Full Time,Senior,New York City,USD,149000,137500.0,1.000000


In [12]:
# Group by ‘experience_level’ -> calculate the average and median salary:
ds_salaries.groupby('experience_level')['salary'].agg(['mean','median'])

,mean,median
experience_level,,
Entry,36111.111111,30000.0
Executive,76076.923077,46000.0
Mid,51786.885246,51000.0
Senior,75088.033012,68000.0


In [11]:
# PCA
numeric_col=[]
for col in ds_salaries.columns:
  print(f"unique counts for {col}: {ds_salaries[col].value_counts()}\n")

unique counts for Unnamed: 0: Unnamed: 0
2259    1
0       1
2       1
2190    1
2189    1
       ..
8       1
7       1
6       1
5       1
4       1
Name: count, Length: 1171, dtype: int64

unique counts for job_title: job_title
Data scientist      394
Data analyst        368
Machine learning    289
Big data            101
ML Ops               19
Name: count, dtype: int64

unique counts for job_type: job_type
Full Time     1136
Internship      35
Name: count, dtype: int64

unique counts for experience_level: experience_level
Senior       727
Mid          305
Entry        126
Executive     13
Name: count, dtype: int64

unique counts for location: location
London                    75
Remote                    50
San Francisco             43
Bengaluru                 34
Paris                     33
                          ..
United States (Remote)     1
Australia                  1
Melbourne                  1
Vancouver                  1
Paris | Remote             1
Name: count, Len

In [58]:
sorted_location_counts = ds_salaries.groupby('location')['location'].count().sort_values(ascending=False).apply(filter)
print(sorted_location_counts[:])

NameError: name 'x' is not defined

In [55]:
sorted_location_counts

,location
location,
London,75
Remote,50
San Francisco,43
Bengaluru,34
Paris,33
...,...
Utrecht,1
Vancouver,1
Vienna,1


In [10]:
ds_salaries.columns

Index(['Unnamed: 0', 'job_title', 'job_type', 'experience_level', 'location',
       'salary_currency', 'salary', 'salary_normalized'],
      dtype='object')

For the PCA and dimensionality reduciton we need nymeric values. Provided is only 1 numeric column - salary. Rest are labeled, qualitative data but with relatively low cardinality, except for location where we have 320 unique locations.

Currency has only 3 values, and overwhlemingly 'USD' (>98%) - suggest to focus the analysis on this currency only.

Unnamed '0' column is useless, seems like a filtered index from original data set -> we drop.

Low cardinality variables like experience_level we do one-hot encoding. High cardinality like location we apply target_encoding (salary group-average)

In [61]:
# first, target_encoding of the location
avg_salary_by_locaiton = ds_salaries.groupby('location')['salary_cap_normalized'].agg(['mean'])
ds_salaries['location_avg_salary'] = ds_salaries['location'].map(avg_salary_by_locaiton['mean'])
ds_salaries['location_avg_salary'].value_counts()

,count
location_avg_salary,
0.240000,75
0.390512,50
0.353488,44
0.727204,43
0.306977,40
...,...
0.167442,1
0.790698,1
0.613953,1


In [62]:
# now one-hot encoding of low cardinality features:
low_card_features = ['experience_level', 'job_type', 'salary_currency','job_title']
ds_salaries = pd.get_dummies(ds_salaries, columns = low_card_features)

,Unnamed: 0,location,salary,salary_cap,salary_cap_normalized,location_avg_salary,experience_level_Entry,experience_level_Executive,experience_level_Mid,experience_level_Senior,job_type_Full Time,job_type_Internship,salary_currency_EUR,salary_currency_GBP,salary_currency_USD,job_title_Big data,job_title_Data analyst,job_title_Data scientist,job_title_ML Ops,job_title_Machine learning
0,0,New York City,149000,137500.0,1.000000,0.587936,False,False,False,True,True,False,False,False,True,False,False,True,False,False
1,2,Boston,120000,120000.0,0.837209,0.627907,False,False,False,True,True,False,False,False,True,False,False,True,False,False
2,3,London,68000,68000.0,0.353488,0.240000,False,False,False,True,True,False,False,False,True,False,False,True,False,False
3,4,Boston,120000,120000.0,0.837209,0.627907,False,False,False,True,True,False,False,False,True,False,False,True,False,False
4,5,New York City,149000,137500.0,1.000000,0.587936,False,False,False,True,True,False,False,False,True,False,False,True,False,False


In [64]:
# drop not needed columns Unnamed:0 and location which was target_encoded:
ds_salaries.drop(columns=['Unnamed: 0', 'location','salary_cap','salary'], inplace=True)
# new target variable is salary_cap_normalized
ds_salaries.head()

KeyError: "['Unnamed: 0', 'location'] not found in axis"

,salary_cap_normalized,location_avg_salary,experience_level_Entry,experience_level_Executive,experience_level_Mid,experience_level_Senior,job_type_Full Time,job_type_Internship,salary_currency_EUR,salary_currency_GBP,salary_currency_USD,job_title_Big data,job_title_Data analyst,job_title_Data scientist,job_title_ML Ops,job_title_Machine learning
0,1.000000,0.587936,False,False,False,True,True,False,False,False,True,False,False,True,False,False
1,0.837209,0.627907,False,False,False,True,True,False,False,False,True,False,False,True,False,False
2,0.353488,0.240000,False,False,False,True,True,False,False,False,True,False,False,True,False,False
3,0.837209,0.627907,False,False,False,True,True,False,False,False,True,False,False,True,False,False
4,1.000000,0.587936,False,False,False,True,True,False,False,False,True,False,False,True,False,False


In [66]:
ds_salaries.shape

(1171, 16)

In [68]:
# now the PCA itself:
from sklearn.decomposition import PCA

#This line creates an instance of the PCA class with the parameter n_components.
# It specifies that we want to reduce the dimensionality of the data to n_components principal components.
# In other words, the data will be projected into a n_components-D space.

# We iterate through a set of reasonable n_component values and select one where most of variance is explained
for n in range(2, 10):
  print(f"n_components: {n}")
  pca = PCA(n_components = n)
  pca.fit(ds_salaries)
  pca_features = pca.fit_transform(ds_salaries)
  print("Original shape:", ds_salaries.shape)
  print("Reduced shape:", pca_features.shape)

  print("Explained variance ratio:", pca.explained_variance_ratio_)
  print("Total explained variance:", sum(pca.explained_variance_ratio_))
  print("\n")

n_components: 2
Original shape: (1171, 16)
Reduced shape: (1171, 2)
Explained variance ratio: [0.28360742 0.22244179]
Total explained variance: 0.5060492033386499


n_components: 3
Original shape: (1171, 16)
Reduced shape: (1171, 3)
Explained variance ratio: [0.28360742 0.22244179 0.18509333]
Total explained variance: 0.6911425328863399


n_components: 4
Original shape: (1171, 16)
Reduced shape: (1171, 4)
Explained variance ratio: [0.28360742 0.22244179 0.18509333 0.10352912]
Total explained variance: 0.7946716496740129


n_components: 5
Original shape: (1171, 16)
Reduced shape: (1171, 5)
Explained variance ratio: [0.28360742 0.22244179 0.18509333 0.10352912 0.07494494]
Total explained variance: 0.8696165924099524


n_components: 6
Original shape: (1171, 16)
Reduced shape: (1171, 6)
Explained variance ratio: [0.28360742 0.22244179 0.18509333 0.10352912 0.07494494 0.04543638]
Total explained variance: 0.9150529741554606


n_components: 7
Original shape: (1171, 16)
Reduced shape: (1171, 

Hence, we can see the n_components 6 or 7 is the best choice as both already explain > 90% of total variance, and after 7 there is barely incremental improvement. That' still a good reduction compared to the normalized, encoded matrix of 16 features.